In [ ]:
#%pip install plotly pandas numpy requests pvlib
#%pip install ipywidgets
import pandas as pd
import numpy as np
import requests
from datetime import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pvlib.location import Location
from pvlib.pvsystem import PVSystem
from pvlib.modelchain import ModelChain
from pvlib.temperature import TEMPERATURE_MODEL_PARAMETERS
from pvlib.irradiance import erbs

# ────────── CONFIG ──────────
LAT, LON = 11.474729, 99.596890
#LAT, LON = 13.850929802369471, 100.55747605767155 # PEA
ALTITUDE = 50
SYSTEM_KW_DC = 5000
TILT, AZIMUTH = 15, 180

LOSSES = {
    'soiling': 0,
    'shading': 0,
    'mismatch': 0,
    'wiring': 0,
    'connections': 0,
    'lid': 0,
    'nameplate': 0,
    'availability': 0,
    'degradation': 7.38
}

def get_weather_open_meteo(day):
    today = datetime.now().date()
    target = datetime.strptime(day, "%Y-%m-%d").date()
    base = "https://archive-api.open-meteo.com/v1/archive" if target < today else "https://api.open-meteo.com/v1/forecast"
    params = ["temperature_2m","cloud_cover","direct_normal_irradiance","diffuse_radiation","wind_speed_10m"]
    url = f"{base}?latitude={LAT}&longitude={LON}&hourly={','.join(params)}&start_date={day}&end_date={day}&timezone=Asia%2FBangkok"
    data = requests.get(url, timeout=20).json()
    h = data["hourly"]
    times = pd.to_datetime(h["time"]).tz_localize("Asia/Bangkok")
    wx = pd.DataFrame({
        'temp_air':    h["temperature_2m"],
        'cloud_cover': h.get("cloud_cover", [0]*len(times)),
        'dni':         h["direct_normal_irradiance"],
        'dhi':         h.get("diffuse_radiation", [0]*len(times)),
        'wind_speed':  h["wind_speed_10m"]
    }, index=times)
    solpos = Location(LAT, LON).get_solarposition(times)
    zen = solpos['zenith']
    wx['ghi'] = (wx['dni'] * np.cos(np.radians(zen)) + wx['dhi']).clip(lower=0)
    if wx['dhi'].sum() == 0:
        decomp = erbs(wx['ghi'], zen, times)
        wx['dhi'], wx['dni'] = decomp['dhi'], decomp['dni']
    return wx

def calculate_dynamic_loss(wx):
    """ปรับ shading loss ตาม avg cloud cover ของวันนั้น"""
    losses = LOSSES.copy()
    avg_cloud = wx['cloud_cover'].mean() / 100.0 
    losses['shading'] *= (1 + avg_cloud)        
    eff = np.prod([(1 - v/100) for v in losses.values()])
    return (1 - eff) * 100

def simulate_pv(wx):
    eff_factor = 0.9
    system = PVSystem(
        surface_tilt=TILT,
        surface_azimuth=AZIMUTH,
        module_parameters={'pdc0': SYSTEM_KW_DC * eff_factor, 'gamma_pdc': -0.004},
        inverter_parameters={'pdc0': SYSTEM_KW_DC * eff_factor * 1.05, 'eta_inv_nom': 0.97},
        temperature_model_parameters=TEMPERATURE_MODEL_PARAMETERS['sapm']['open_rack_glass_glass']
    )
    loc = Location(LAT, LON, altitude=ALTITUDE)
    mc = ModelChain(system, loc, dc_model='pvwatts', ac_model='pvwatts', aoi_model='no_loss')
    mc.run_model(wx)

    loss_pct = calculate_dynamic_loss(wx)
    ac_net = (mc.results.ac * (1 - loss_pct/100)).clip(lower=0)

    return pd.DataFrame({
        'dc_power':     mc.results.dc,
        'ac_power_raw': mc.results.ac,
        'ac_power_net': ac_net,
        'cell_temp':    mc.results.cell_temperature,
        'ghi':          wx['ghi'],
        'dni':          wx['dni']
    }, index=wx.index)

def main():
    date_input = input("Enter date (YYYY-MM-DD): ")
    wx = get_weather_open_meteo(date_input)
    res = simulate_pv(wx)

    daily_kwh = res['ac_power_net'].sum() / 1000
    peak_kw   = res['ac_power_net'].max()
    cf        = daily_kwh / (SYSTEM_KW_DC * 24 / 1000) * 100
    loss_pct  = calculate_dynamic_loss(wx)

    fig = make_subplots(rows=2, cols=1, subplot_titles=("AC Power (Net vs Raw)", "Solar Irradiance"), shared_xaxes=True)
    fig.add_trace(go.Scatter(x=res.index, y=res['ac_power_net'], name='AC Net', fill='tozeroy'), 1, 1)
    fig.add_trace(go.Scatter(x=res.index, y=res['ac_power_raw'], name='AC Raw', line=dict(dash='dash')), 1, 1)
    fig.add_trace(go.Scatter(x=res.index, y=res['ghi'], name='GHI'), 2, 1)
    fig.add_trace(go.Scatter(x=res.index, y=res['dni'], name='DNI', line=dict(dash='dot')), 2, 1)

    fig.update_layout(
        title=f"Solar Forecast {date_input} | Daily {daily_kwh:.1f} kWh | Peak {peak_kw:.0f} kW | CF {cf:.1f}% | Loss {loss_pct:.1f}%",
        hovermode="x unified", height=700
    )
    out = f"solar_forecast_{date_input}.html"
    fig.write_html(out, auto_open=True)
    print(f"✅ Saved: {out}")

if __name__ == "__main__":
    main()

In [ ]:
# %pip install plotly pandas numpy requests pvlib ipywidgets

import pandas as pd
import numpy as np
import requests
from datetime import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pvlib.location import Location
from pvlib.pvsystem import PVSystem
from pvlib.modelchain import ModelChain
from pvlib.temperature import TEMPERATURE_MODEL_PARAMETERS
from pvlib.irradiance import erbs
# ────────── CONFIG ──────────
#LAT, LON      = 11.474729, 99.596890 #กรอก ละติจูด ลองจิจูด
LAT, LON    = 13.850929802369471, 100.55747605767155  # PEA
ALTITUDE      = 50 #ความสูงจากระดับน้ำทะเล
SYSTEM_KW_DC  = 5000  #ความจุโรงไฟฟ้า    
TILT, AZIMUTH = 15, 180  
#TILT, AZIMUTH = 15, 180  #มุมเอียงและทิศทางของแผงโซล่าเซลล์
LOSSES = {
    'soiling':      0,
    'shading':      0,
    'mismatch':     0,
    'wiring':       0,
    'connections':  0,
    'lid':          0,
    'nameplate':    0,
    'availability': 0,
    'degradation':  12
}
#-------------------------------------------------
def get_weather_open_meteo(day: str) -> pd.DataFrame: 
    today  = datetime.now().date()
    target = datetime.strptime(day, "%Y-%m-%d").date()
    base   = ("https://archive-api.open-meteo.com/v1/archive" #ข้อมูล อดีต
              if target < today else
              "https://api.open-meteo.com/v1/forecast") # forecast อนาคต
    params = ["temperature_2m","cloud_cover","direct_normal_irradiance",
              "diffuse_radiation","wind_speed_10m"]
    url = (
        f"{base}?latitude={LAT}&longitude={LON}"
        f"&hourly={','.join(params)}"
        f"&start_date={day}&end_date={day}"
        f"&timezone=Asia%2FBangkok"
    )
    data = requests.get(url, timeout=20).json()
    h    = data["hourly"]
    times = pd.to_datetime(h["time"]).tz_localize("Asia/Bangkok")

    wx = pd.DataFrame({
        'temp_air':    h["temperature_2m"],
        'cloud_cover': h.get("cloud_cover", [0]*len(times)),
        'dni':         h["direct_normal_irradiance"],
        'dhi':         h.get("diffuse_radiation", [0]*len(times)),
        'wind_speed':  h["wind_speed_10m"]
    }, index=times)

    solpos = Location(LAT, LON).get_solarposition(times)
    zen    = solpos['zenith']
    wx['ghi'] = (wx['dni'] * np.cos(np.radians(zen)) + wx['dhi']).clip(lower=0)

    if wx['dhi'].sum() == 0:
        decomp    = erbs(wx['ghi'], zen, times)
        wx['dhi'] = decomp['dhi']
        wx['dni'] = decomp['dni']

    return wx

def calculate_dynamic_loss(wx: pd.DataFrame) -> float:
    """ปรับ shading loss ตาม avg cloud cover ของวันนั้น"""
    losses    = LOSSES.copy()
    avg_cloud = wx['cloud_cover'].mean() / 100.0
    losses['shading'] *= (1 + avg_cloud)
    eff       = np.prod([(1 - v/100) for v in losses.values()])
    return (1 - eff) * 100

def simulate_pv(wx: pd.DataFrame) -> pd.DataFrame:
    eff = 0.9
    system = PVSystem(
        surface_tilt=TILT,
        surface_azimuth=AZIMUTH,
        module_parameters={'pdc0': SYSTEM_KW_DC * eff, 'gamma_pdc': -0.004},
        inverter_parameters={'pdc0': SYSTEM_KW_DC * eff * 1.05, 'eta_inv_nom': 0.97},
        temperature_model_parameters=TEMPERATURE_MODEL_PARAMETERS['sapm']['open_rack_glass_glass']
    )
    loc = Location(LAT, LON, altitude=ALTITUDE)
    mc  = ModelChain(system, loc, dc_model='pvwatts', ac_model='pvwatts', aoi_model='no_loss')
    mc.run_model(wx)

    loss_pct = calculate_dynamic_loss(wx)
    ac_net   = (mc.results.ac * (1 - loss_pct/100)).clip(lower=0)

    return pd.DataFrame({
        'dc_power':     mc.results.dc,
        'ac_power_raw': mc.results.ac,
        'ac_power_net': ac_net,
        'cell_temp':    mc.results.cell_temperature,
        'ghi':          wx['ghi'],
        'dni':          wx['dni']
    }, index=wx.index)

def main():
    date_input = input("Enter date (YYYY-MM-DD): ").strip() #รับวันที่จากผู้ใช้
    wx         = get_weather_open_meteo(date_input)
    res        = simulate_pv(wx)

    total_kwh = res['ac_power_net'].sum() / 1000.0
    peak_kw = res['ac_power_net'].max()
    cf = total_kwh / (SYSTEM_KW_DC * 24 / 1000) * 100
    total_kWh = ((res['ac_power_net']).sum()) * 2
    peak_wto_kw = peak_kw / 1000
    total_wto_kw =  total_kWh / 1000.0

    loss_pct = calculate_dynamic_loss(wx)

    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=("AC Power (Net vs Raw)", "Solar Irradiance"),
        shared_xaxes=True
    )
    fig.add_trace(go.Scatter(
        x=res.index, y=res['ac_power_net'], name='AC Net', fill='tozeroy'
    ), 1, 1)
    fig.add_trace(go.Scatter(
        x=res.index, y=res['ac_power_raw'], name='AC Raw', line=dict(dash='dash')
    ), 1, 1)
    fig.add_trace(go.Scatter(
        x=res.index, y=res['ghi'], name='GHI'
    ), 2, 1)
    fig.add_trace(go.Scatter(
        x=res.index, y=res['dni'], name='DNI', line=dict(dash='dot')
    ), 2, 1)

    fig.update_layout(
        title=(
            f"Solar Forecast {date_input} | "
            f"Peak {peak_kw:.2f}Kw ({peak_wto_kw:.2f}Mw) | "
            f"CF {cf:.2f}% | "
            f"Loss {loss_pct:.2f}%  | "
            f"Total {total_kWh:.2f}Kw ({total_wto_kw:.2f}Mw)"
        ),
        hovermode="x unified",
        height=700
    )

    out = f"solar_forecast_{date_input}.html"
    fig.write_html(out, auto_open=True)
    print(f"✅ Saved: {out}")

if __name__ == "__main__":
    main()